# Iššūkis: Analizuojame tekstą apie duomenų mokslą

Šiame pavyzdyje atliksime paprastą pratimą, apimantį visus tradicinio duomenų mokslo proceso žingsnius. Jums nereikia rašyti jokio kodo, galite tiesiog spustelėti žemiau esančius langelius, kad juos vykdytumėte ir stebėtumėte rezultatą. Kaip iššūkį, jums siūloma išbandyti šį kodą su įvairiais duomenimis.

## Tikslas

Šioje pamokoje aptarėme skirtingas su duomenų mokslu susijusias sąvokas. Pabandykime atrasti daugiau susijusių sąvokų atlikdami **teksto gavybą**. Pradėsime nuo teksto apie duomenų mokslą, ištrauksime iš jo raktinius žodžius, o tada bandysime vizualizuoti rezultatą.

Kaip tekstą naudosiu puslapį apie duomenų mokslą iš Vikipedijos:


In [ ]:
url = 'https://en.wikipedia.org/wiki/Data_science'

## 1 veiksmas: Duomenų gavimas

Pirmas žingsnis kiekviename duomenų mokslo procese yra duomenų gavimas. Tam naudosime `requests` biblioteką:


In [ ]:
import requests

# Define a custom header.
headers = {
    'User-Agent': 'DataScienceChallenge/1.0 (myemail@gmail.com)'
}

# Pass the headers into the get request
response = requests.get(url, headers=headers)

if response.status_code == 200:
    text = response.content.decode('utf-8')
    print(text[:1000])
else:
    print(f"Error: {response.status_code}")

## 2 veiksmas: Duomenų transformavimas

Kitas žingsnis – paversti duomenis apdorojimui tinkama forma. Mūsų atveju atsisiuntėme HTML šaltinio kodą iš puslapio ir jį reikia konvertuoti į paprastą tekstą.

Tam galima pasitelkti įvairius metodus. Naudosime [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/), populiarią Python biblioteką HTML analizavimui. BeautifulSoup leidžia mums taikyti konkrečius HTML elementus, todėl galime susitelkti į pagrindinį Wikipedia straipsnio turinį ir sumažinti navigacijos meniu, šoninių juostų, poraščių ir kito nereikalingo turinio kiekį (nors kai kurie standartiniai teksto elementai vis dar gali likti).


Pirmiausia turime įdiegti BeautifulSoup biblioteką HTML analizavimui:


In [ ]:
import sys
!{sys.executable} -m pip install beautifulsoup4

In [ ]:
from bs4 import BeautifulSoup

# Parse the HTML content
soup = BeautifulSoup(text, 'html.parser')

# Extract only the main article content from Wikipedia
# Wikipedia uses 'mw-parser-output' class for the main article content
content = soup.find('div', class_='mw-parser-output')

def clean_wikipedia_content(content_node):
    """Remove common non-article elements from a Wikipedia content node."""
    # Strip jump links, navboxes, reference lists/superscripts, edit sections, TOC, sidebars, etc.
    selectors = [
        '.mw-jump-link',
        '.navbox',
        '.reflist',
        'sup.reference',
        '.mw-editsection',
        '.hatnote',
        '.metadata',
        '.infobox',
        '#toc',
        '.toc',
        '.sidebar',
    ]
    for selector in selectors:
        for el in content_node.select(selector):
            el.decompose()

if content:
    # Clean the content node to better approximate article text only.
    clean_wikipedia_content(content)
    text = content.get_text(separator=' ', strip=True)
    print(text[:1000])
else:
    print("Could not find main content. Using full page text.")
    text = soup.get_text(separator=' ', strip=True)
    print(text[:1000])

## 3 žingsnis: Gauti įžvalgas

Svarbiausias žingsnis yra paversti mūsų duomenis į tam tikrą formą, iš kurios galėtume gauti įžvalgas. Mūsų atveju norime iš teksto išgauti raktinius žodžius ir pamatyti, kurie raktiniai žodžiai yra reikšmingesni.

Naudosime Python biblioteką, vadinamą [RAKE](https://github.com/aneesha/RAKE), raktinių žodžių išgavimui. Pirmiausia įsidiekime šią biblioteką, jei jos nėra:


In [ ]:
import sys
!{sys.executable} -m pip install nlp_rake

Pagrindinė funkcionalumo dalis yra prieinama iš `Rake` objekto, kurį galime pritaikyti naudodami tam tikrus parametrus. Mūsų atveju nustatysime raktinio žodžio minimalią ilgį – 5 simbolius, minimalią raktinio žodžio dažnį dokumente – 3, o maksimalią žodžių raktiniame žodyje skaičių – 2. Drąsiai eksperimentuokite su kitomis vertėmis ir stebėkite rezultatą.


In [ ]:
import nlp_rake
extractor = nlp_rake.Rake(max_words=2,min_freq=3,min_chars=5)
res = extractor.apply(text)
res


Mes gavome terminų sąrašą kartu su susijusiu svarbos laipsniu. Kaip matote, labiausiai aktualios disciplinos, tokios kaip mašininis mokymasis ir didieji duomenys, yra sąrašo viršutinėse pozicijose.

## 4 žingsnis: rezultato vizualizavimas

Žmonės geriausiai interpretuoja duomenis vizualioje formoje. Todėl dažnai yra prasminga vizualizuoti duomenis, kad būtų galima gauti įžvalgų. Galime naudoti Python `matplotlib` biblioteką, kad nubrėžtume paprastą raktinių žodžių su jų aktualumu pasiskirstymą:


In [ ]:
import matplotlib.pyplot as plt

def plot(pair_list):
    k,v = zip(*pair_list)
    plt.bar(range(len(k)),v)
    plt.xticks(range(len(k)),k,rotation='vertical')
    plt.show()

plot(res)

Tačiau yra dar geresnis būdas vizualizuoti žodžių dažnumą – naudojant **Žodžių Debesį**. Mums reikės įdiegti kitą biblioteką, kad galėtume nubraižyti žodžių debesį iš mūsų raktinių žodžių sąrašo.


In [ ]:
!{sys.executable} -m pip install wordcloud

`WordCloud` objektas atsakingas už originalaus teksto arba iš anksto apskaičiuoto žodžių sąrašo su jų dažniais priėmimą ir grąžina vaizdą, kurį galima parodyti naudojant `matplotlib`:


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

wc = WordCloud(background_color='white',width=800,height=600)
plt.figure(figsize=(15,7))
plt.imshow(wc.generate_from_frequencies({ k:v for k,v in res }))

Taip pat galime perduoti originalų tekstą `WordCloud` - pažiūrėkime, ar sugebėsime gauti panašų rezultatą:


In [ ]:
plt.figure(figsize=(15,7))
plt.imshow(wc.generate(text))

In [ ]:
wc.generate(text).to_file('images/ds_wordcloud.png')

Matote, kad žodžių debesėlis dabar atrodo įspūdingiau, tačiau jame taip pat yra daug triukšmo (pvz., nesusiję žodžiai, tokie kaip `Retrieved on`). Be to, gauname mažiau raktinių žodžių, sudarytų iš dviejų žodžių, pavyzdžiui, *data scientist* arba *computer science*. Taip yra todėl, kad RAKE algoritmas geriau atlieka gerų raktinių žodžių atrinkimą iš teksto. Šis pavyzdys iliustruoja duomenų priešapdorojimo ir valymo svarbą, nes aiškus vaizdas galutiniame rezultate leis priimti geresnius sprendimus.

Šiame pratime mes praėjome paprastą procesą, kaip iš Wikipedijos teksto išgauti prasmę raktinių žodžių ir žodžių debesėlio forma. Šis pavyzdys yra gana paprastas, bet gerai demonstruoja visus įprastus žingsnius, kuriuos duomenų mokslininkas atlieka dirbdamas su duomenimis, pradedant nuo duomenų gavimo iki vizualizacijos.

Mūsų kurse aptarsime visus šiuos žingsnius išsamiai.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, prašome atkreipti dėmesį, kad automatiniai vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas jo gimtąja kalba laikomas autoritetingu šaltiniu. Svarbiai informacijai rekomenduojama naudoti profesionalų žmogiškąjį vertimą. Mes neatsakome už jokius nesusipratimus ar neteisingą interpretaciją, kilusią naudojantis šiuo vertimu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
